# VisionBridge — trained model check (Colab)

This notebook is **inference/checking only**. It does not train, resume training, download the dataset, or install MediaPipe.

Goal: verify an already-trained `base_model.pt` + vocabulary, then run the model on a real 132-dim pose + 1404-dim face keypoint sequence.


## 1. Bootstrap the repository import path

Run this cell first. It explicitly adds `/content/VisionBridge/backend` to `sys.path`. Later cells repeat this bootstrap defensively, so they also work when executed individually.


In [ ]:
import os, sys, subprocess
from pathlib import Path

BASE = Path('/content')
REPO_ROOT = BASE / 'VisionBridge'

if not (REPO_ROOT / 'README.md').exists():
    subprocess.run(['git', 'clone', 'https://github.com/BharathWaj-K-R/VisionBridge.git', str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))
os.chdir(REPO_ROOT)

print('Repository:', REPO_ROOT)
print('Backend import root:', BACKEND_ROOT)
print('Current directory:', Path.cwd())
assert (BACKEND_ROOT / 'app' / '__init__.py').exists(), 'backend/app is missing'
import app
print('APP IMPORT: PASS')


## 2. Locate the trained checkpoint and vocabulary

Expected paths:
`backend/app/models/weights/base_model.pt`
`backend/app/models/weights/base_model.vocab.json`


In [ ]:
from pathlib import Path
import shutil, sys

BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

WEIGHTS = REPO_ROOT / 'backend' / 'app' / 'models' / 'weights' / 'base_model.pt'
VOCAB = REPO_ROOT / 'backend' / 'app' / 'models' / 'weights' / 'base_model.vocab.json'

if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    print('Upload BOTH base_model.pt and base_model.vocab.json')
    uploaded = files.upload()
    for name in ('base_model.pt', 'base_model.vocab.json'):
        if name not in uploaded:
            raise FileNotFoundError(f'Missing required artifact: {name}')
        shutil.copy(name, WEIGHTS.parent / name)

assert WEIGHTS.exists(), f'Missing weights: {WEIGHTS}'
assert VOCAB.exists(), f'Missing vocabulary: {VOCAB}'
print(f'Weights: {WEIGHTS} ({WEIGHTS.stat().st_size/1e6:.2f} MB)')
print(f'Vocab:   {VOCAB} ({VOCAB.stat().st_size/1e3:.2f} KB)')


## 3. Validate the trained checkpoint


In [ ]:
import sys, torch
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.training.isltranslate import SimpleCharTokenizer
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH

tokenizer = SimpleCharTokenizer.load(VOCAB)
state = torch.load(WEIGHTS, map_location='cpu')
assert isinstance(state, dict), 'Checkpoint is not a state-dict dictionary.'
assert 'output_head.weight' in state, 'Checkpoint does not contain output_head.weight.'
checkpoint_vocab = int(state['output_head.weight'].shape[0])
print('Checkpoint vocabulary:', checkpoint_vocab)
print('Tokenizer vocabulary: ', tokenizer.vocab_size)
assert checkpoint_vocab == tokenizer.vocab_size, 'Checkpoint/vocabulary size mismatch.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device).eval()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Device:', device)
print('Trainable parameters:', trainable)
assert trainable == 0, 'Base model is not frozen.'
print('CHECKPOINT VALIDATION: PASS')


## 4. Forward-pass smoke test

Synthetic zeros are used only to verify the model's tensor contract. This is **not** an accuracy test.


In [ ]:
import torch, sys
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

frames = 16
pose = torch.zeros(1, frames, POSE_INPUT_DIM, device=device)
face = torch.zeros(1, frames, FACE_INPUT_DIM, device=device)
with torch.inference_mode():
    logits = model(pose, face)
print('Pose:', tuple(pose.shape))
print('Face:', tuple(face.shape))
print('Logits:', tuple(logits.shape))
assert logits.shape == (1, frames, tokenizer.vocab_size), f'Unexpected logits shape: {tuple(logits.shape)}'
print('FORWARD TEST: PASS')


## 5. Real model prediction from processed keypoints

This is the important test. It uses a real pose/face sequence and the same `decode_logits()` decoder used by VisionBridge inference.


In [ ]:
import sys, numpy as np, torch
from pathlib import Path
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.services.inference_service import decode_logits
from app.training.isltranslate import ISLTranslateKeypointDataset, _downsample_to_max_length

DATA_DIR = REPO_ROOT / 'data' / 'processed' / 'isltranslate'
if (DATA_DIR / 'ISLTranslate.csv').exists():
    dataset = ISLTranslateKeypointDataset(DATA_DIR, tokenizer=tokenizer)
    assert len(dataset) > 0, 'No usable processed samples found.'
    item = dataset[0]
    pose, face = _downsample_to_max_length(item['pose'], item['face'], item['uid'])
    with torch.inference_mode():
        logits = model(pose.unsqueeze(0).to(device), face.unsqueeze(0).to(device))
    prediction, confidence = decode_logits(logits)
    print('UID:', item['uid'])
    print('GROUND TRUTH:', item['text'])
    print('PREDICTED:   ', prediction)
    print('CONFIDENCE:  ', round(float(confidence), 4))
    print('FRAMES USED:', pose.shape[0])
    print('REAL SAMPLE INFERENCE: PASS')
else:
    print('No processed dataset found at:', DATA_DIR)
    print('Upload pose.npy + face.npy in the next cell instead.')


## 6. Direct prediction from uploaded pose.npy + face.npy

Use this when you already have extracted keypoints but do not want to download the dataset. Upload files named like `pose.npy` and `face.npy`.


In [ ]:
import sys, numpy as np, torch
BACKEND_ROOT = REPO_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.services.inference_service import decode_logits
from app.training.isltranslate import _downsample_to_max_length

from google.colab import files
print('Upload pose.npy and face.npy')
uploaded = files.upload()
pose_files = [n for n in uploaded if 'pose' in n.lower() and n.lower().endswith('.npy')]
face_files = [n for n in uploaded if 'face' in n.lower() and n.lower().endswith('.npy')]
assert pose_files and face_files, 'Please upload files named like pose.npy and face.npy.'

pose_np = np.load(pose_files[0])
face_np = np.load(face_files[0])
assert pose_np.ndim == 2 and pose_np.shape[1] == POSE_INPUT_DIM
assert face_np.ndim == 2 and face_np.shape[1] == FACE_INPUT_DIM
assert pose_np.shape[0] == face_np.shape[0] and pose_np.shape[0] > 0
pose_t, face_t = _downsample_to_max_length(torch.from_numpy(pose_np).float(), torch.from_numpy(face_np).float(), 'uploaded')
with torch.inference_mode():
    logits = model(pose_t.unsqueeze(0).to(device), face_t.unsqueeze(0).to(device))
prediction, confidence = decode_logits(logits)
print('POSE SHAPE:', pose_np.shape)
print('FACE SHAPE:', face_np.shape)
print('PREDICTED TEXT:', prediction)
print('CONFIDENCE:', round(float(confidence), 4))
print('UPLOADED-KEYPOINT INFERENCE: PASS')


## Final interpretation

**PASS means:** the trained checkpoint loads, vocabulary matches, the model accepts the expected inputs, the forward pass works, and the real decoder produces text.

It does **not** prove model accuracy. For accuracy, use a held-out labeled set and calculate CER/WER separately.
